# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barak-hub/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

### Research Title
Predicting Google Search Engagement Opportunities Using Machine Learning

### Research Question
Can historical Google Search performance signals help identify content opportunities with low click-through rates (CTR), and can machine learning improve opportunity ranking compared with a simple rule-based baseline?

### Problem Statement
Some content receives many impressions in Google Search but relatively few clicks. This may indicate opportunities for improvement, although low CTR can also result from ranking position, search intent, or query relevance.

### Decision Supported
This research aims to help SEO specialists prioritize which content should be investigated for potential improvement.

### Proposed Approach
I will use FlyRank's anonymized search performance data to investigate engagement patterns, develop a machine learning approach, and compare it against the rule-based baseline created in ML-07.

### Expected Value
The project aims to provide evidence-based recommendations that support SEO decision-making. Any improvement over the baseline must be demonstrated through evaluation rather than assumed.

### Data Description

This study uses FlyRank's anonymized `fact_content_query_90d.parquet` dataset from the FlyRank Internship Warehouse on Hugging Face.

The dataset contains 2,414,248 records and 21 columns, covering April 2 to June 30, 2026. Each record represents an anonymized client–content–query combination.

The data includes Google Search impressions, clicks, average ranking position, query-length features, and previous and latest 30-day performance metrics.

### Initial Data Quality Findings

- The dataset represents 52 anonymized clients.
- No missing values were found in the seven search-performance columns checked.
- No records had zero impressions over the 90-day reporting period.
- The dataset contains a single 90-day reporting window.

### Data Privacy

Client identities, website URLs, and private search queries will not be published. Only anonymized identifiers, aggregated statistics, and public-safe findings will be used.

### Limitations

The dataset contains aggregated historical observations rather than individual search events. Ranking position, search intent, and other unobserved factors may influence CTR. The overlapping 30-day and 90-day metrics must be handled carefully to avoid data leakage.

In [1]:
import pandas as pd

from google.colab import userdata
from huggingface_hub import hf_hub_download

token = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=token
)

df = pd.read_parquet(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDate windows:")
print(df[["window_start", "window_end"]].drop_duplicates().head(10))

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

RuntimeError: Task error: File reconstruction error: CAS Client Error: Request error: HTTP status server error (500 Internal Server Error), domain: https://us.gcp.cdn.hf.co/xorbs/default/e0923e9bc293049262753e2addd828fa0dc0f98051f3ae2eabf0afb9cfecbfc3

In [ ]:
# Check missing values
important_columns = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "impressions_prev30",
    "clicks_prev30",
    "impressions_last30",
    "clicks_last30"
]

print("Missing values:")
print(df[important_columns].isna().sum())

print("\nZero-impression records:")
print((df["impressions_90d"] == 0).sum())

print("\nUnique clients:")
print(df["client_hash_id"].nunique())

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:

# Examine historical and latest 30-day performance

columns = [
    "impressions_prev30",
    "clicks_prev30",
    "impressions_last30",
    "clicks_last30",
    "avg_position_prev30",
    "avg_position_last30"
]

print("30-day performance summary:")
print(df[columns].describe().round(2).to_string())

print("\nRecords with impressions in both periods:")
print(
    (
        (df["impressions_prev30"] > 0) &
        (df["impressions_last30"] > 0)
    ).sum()
)


In [ ]:

# Examine the potential prediction target

import numpy as np

eligible = df[
    (df["impressions_prev30"] > 0) &
    (df["impressions_last30"] > 0)
].copy()

eligible["ctr_prev30"] = (
    eligible["clicks_prev30"] /
    eligible["impressions_prev30"]
)

eligible["ctr_last30"] = (
    eligible["clicks_last30"] /
    eligible["impressions_last30"]
)

print("Eligible records:", len(eligible))

print("\nPrevious and latest CTR summary:")
print(
    eligible[["ctr_prev30", "ctr_last30"]]
    .describe()
    .round(4)
)

print("\nRecords with zero clicks in latest period:")
print((eligible["clicks_last30"] == 0).sum())

print("\nRecords with latest CTR above zero:")
print((eligible["ctr_last30"] > 0).sum())


In [ ]:

# Examine how impression volume affects the target

eligible["impression_bucket"] = pd.cut(
    eligible["impressions_last30"],
    bins=[0, 10, 100, 1000, float("inf")],
    labels=["1-10", "11-100", "101-1000", "1001+"],
    include_lowest=True
)

volume_check = (
    eligible.groupby("impression_bucket", observed=True)
    .agg(
        n=("clicks_last30", "size"),
        impressions=("impressions_last30", "sum"),
        clicks=("clicks_last30", "sum"),
        records_with_clicks=("clicks_last30", lambda x: (x > 0).sum())
    )
)

volume_check["click_rate"] = (
    volume_check["records_with_clicks"] / volume_check["n"]
)

volume_check["ctr"] = (
    volume_check["clicks"] / volume_check["impressions"]
)

display(volume_check)


In [ ]:

# Check historical impression volume

historical_volume = pd.DataFrame({
    "minimum_impressions": [1, 10, 30, 50, 100, 500],
    "eligible_records": [
        (eligible["impressions_prev30"] >= threshold).sum()
        for threshold in [1, 10, 30, 50, 100, 500]
    ]
})

historical_volume["percentage"] = (
    historical_volume["eligible_records"] /
    len(eligible) * 100
).round(2)

display(historical_volume)


In [ ]:

# Create the initial modeling dataset

model_df = eligible[
    eligible["impressions_prev30"] >= 30
].copy()

# Latest-period CTR is the potential prediction target
model_df["target_ctr"] = (
    model_df["clicks_last30"] /
    model_df["impressions_last30"]
)

print("Modeling records:", len(model_df))

print("\nTarget CTR summary:")
print(model_df["target_ctr"].describe().round(4))

print("\nRecords with positive CTR:")
print((model_df["target_ctr"] > 0).sum())

print("\nRecords with zero CTR:")
print((model_df["target_ctr"] == 0).sum())


In [ ]:

# Build candidates using historical information only

candidates = df[
    df["impressions_prev30"] >= 30
].copy()

print("Historical candidates:", len(candidates))

print("\nCandidates with zero latest-period impressions:")
print((candidates["impressions_last30"] == 0).sum())

print("\nCandidates with positive latest-period impressions:")
print((candidates["impressions_last30"] > 0).sum())

print("\nPreviously selected modeling records:")
print(len(model_df))


In [ ]:

# Define future outcomes without excluding historical candidates

candidates["future_visible"] = (
    candidates["impressions_last30"] > 0
).astype(int)

candidates["future_ctr"] = np.where(
    candidates["impressions_last30"] > 0,
    candidates["clicks_last30"] /
    candidates["impressions_last30"].replace(0, np.nan),
    np.nan
)

print("Total candidates:", len(candidates))

print("\nFuture visibility counts:")
print(candidates["future_visible"].value_counts())

print("\nDefined future CTR values:")
print(candidates["future_ctr"].notna().sum())

print("\nUndefined future CTR values:")
print(candidates["future_ctr"].isna().sum())


In [ ]:

# Step 8: Prepare historical model features

candidates["ctr_prev30"] = (
    candidates["clicks_prev30"] /
    candidates["impressions_prev30"]
)

feature_columns = [
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "ctr_prev30",
    "query_token_count",
    "query_char_count"
]

X = candidates[feature_columns].copy()

print("Feature matrix shape:", X.shape)

print("\nMissing values by feature:")
print(X.isna().sum())

print("\nSample feature rows:")
display(X.head())


In [ ]:

# Step 9: Examine client groups before splitting

client_counts = (
    candidates.groupby("client_hash_id")
    .size()
    .sort_values(ascending=False)
)

print("Number of clients:", len(client_counts))

print("\nRecords per client:")
print(client_counts.describe().round(2))

print("\nLargest 5 client groups:")
print(client_counts.head())

print("\nSmallest 5 client groups:")
print(client_counts.tail())


In [ ]:

# Step 10: Split by client to prevent client overlap

from sklearn.model_selection import GroupShuffleSplit

groups = candidates["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("\nClients shared between sets:")
print(len(train_clients.intersection(test_clients)))

print("\nTest percentage of records:")
print(round(len(X_test) / len(X) * 100, 2))


In [ ]:

# Step 11: Compare outcome distributions

train_data = candidates.iloc[train_idx].copy()
test_data = candidates.iloc[test_idx].copy()

for name, data in [
    ("Training", train_data),
    ("Testing", test_data)
]:
    print(f"\n{name} dataset")
    print("Records:", len(data))

    print("Future visibility rate:",
          round(data["future_visible"].mean() * 100, 2), "%")

    visible = data[data["future_visible"] == 1]

    print("Records with positive CTR:",
          (visible["future_ctr"] > 0).sum())

    print("Positive CTR rate among visible records:",
          round((visible["future_ctr"] > 0).mean() * 100, 2), "%")

    print("Average future CTR:",
          round(visible["future_ctr"].mean() * 100, 4), "%")


In [ ]:

# Step 12: Majority-class baseline for future visibility

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix
)

y_train = train_data["future_visible"]
y_test = test_data["future_visible"]

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

print("Baseline accuracy:",
      round(accuracy_score(y_test, baseline_predictions), 4))

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, baseline_predictions), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, baseline_predictions))

print("\nPredicted classes:")
print(pd.Series(baseline_predictions).value_counts())


In [ ]:

# Step 13: Train a logistic regression visibility model

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import recall_score

visibility_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
)

visibility_model.fit(X_train, y_train)

visibility_predictions = visibility_model.predict(X_test)

print("Logistic regression accuracy:",
      round(accuracy_score(y_test, visibility_predictions), 4))

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, visibility_predictions), 4))

print("Recall for records losing visibility:",
      round(recall_score(
          y_test,
          visibility_predictions,
          pos_label=0
      ), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, visibility_predictions))


In [ ]:

# Step 14: Evaluate lost-visibility risk scores

from sklearn.metrics import (
    precision_score,
    average_precision_score,
    roc_auc_score
)

# Probability that a record loses visibility (class 0)
lost_visibility_scores = visibility_model.predict_proba(
    X_test
)[:, 0]

# Convert actual outcomes: 1 means visibility was lost
y_lost = (y_test == 0).astype(int)

# Convert model predictions to the same definition
predicted_lost = (visibility_predictions == 0).astype(int)

print("Precision for lost visibility:",
      round(precision_score(y_lost, predicted_lost), 4))

print("Recall for lost visibility:",
      round(recall_score(y_lost, predicted_lost), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_lost, lost_visibility_scores), 4))

print("Average precision (PR-AUC summary):",
      round(average_precision_score(
          y_lost, lost_visibility_scores
      ), 4))

print("Lost-visibility prevalence:",
      round(y_lost.mean(), 4))


In [ ]:

# Step 15: Examine historical CTR opportunities

candidates["historical_low_ctr"] = (
    candidates["ctr_prev30"] < 0.02
)

candidates["historical_opportunity_score"] = (
    candidates["impressions_prev30"] *
    (0.02 - candidates["ctr_prev30"]).clip(lower=0)
)

print("Total historical candidates:", len(candidates))

print("\nHistorical low-CTR records:")
print(candidates["historical_low_ctr"].sum())

print("\nPercentage with low CTR:")
print(round(candidates["historical_low_ctr"].mean() * 100, 2))

print("\nHistorical opportunity score summary:")
print(
    candidates["historical_opportunity_score"]
    .describe()
    .round(2)
)

print("\nTop 5 historical opportunity scores:")
display(
    candidates[
        ["impressions_prev30",
         "clicks_prev30",
         "ctr_prev30",
         "historical_opportunity_score"]
    ]
    .sort_values(
        "historical_opportunity_score",
        ascending=False
    )
    .head()
)


In [ ]:

# Step 16: Examine top historical opportunities

top_opportunities = (
    candidates
    .sort_values(
        "historical_opportunity_score",
        ascending=False
    )
    .head(100)
    .copy()
)

print("Top opportunities:", len(top_opportunities))

print("\nAverage historical impressions:")
print(round(
    top_opportunities["impressions_prev30"].mean(), 2
))

print("\nAverage future impressions:")
print(round(
    top_opportunities["impressions_last30"].mean(), 2
))

print("\nAverage historical CTR (%):")
print(round(
    top_opportunities["ctr_prev30"].mean() * 100, 4
))

print("\nAverage future CTR among visible records (%):")
print(round(
    top_opportunities["future_ctr"].mean() * 100, 4
))

print("\nRecords with no future impressions:")
print((top_opportunities["future_visible"] == 0).sum())


In [ ]:

# Step 17: Compare top 100 with remaining candidates

ranked = candidates.sort_values(
    "historical_opportunity_score",
    ascending=False
).copy()

ranked["group"] = "Other candidates"
ranked.iloc[
    :100,
    ranked.columns.get_loc("group")
] = "Top 100"

comparison = (
    ranked.groupby("group")
    .agg(
        records=("future_visible", "size"),
        historical_impressions=("impressions_prev30", "mean"),
        future_impressions=("impressions_last30", "mean"),
        historical_ctr=("ctr_prev30", "mean"),
        future_ctr=("future_ctr", "mean"),
        future_visibility_rate=("future_visible", "mean")
    )
)

comparison["historical_ctr"] *= 100
comparison["future_ctr"] *= 100
comparison["future_visibility_rate"] *= 100

display(comparison.round(4))


In [ ]:

# Step 18: Evaluate historical ranking on held-out clients

test_ranked = candidates.iloc[test_idx].copy()

# Future click gap relative to the 2% CTR reference
test_ranked["future_click_gap"] = (
    0.02 * test_ranked["impressions_last30"]
    - test_ranked["clicks_last30"]
).clip(lower=0)

# Rank using historical information only
test_ranked = test_ranked.sort_values(
    "historical_opportunity_score",
    ascending=False
)

top_100_test = test_ranked.head(100)

print("Held-out test records:", len(test_ranked))

print("\nTotal future click gap across test records:")
print(round(test_ranked["future_click_gap"].sum(), 2))

print("\nFuture click gap captured by top 100:")
print(round(top_100_test["future_click_gap"].sum(), 2))

print("\nPercentage of future click gap in top 100:")
print(round(
    top_100_test["future_click_gap"].sum() /
    test_ranked["future_click_gap"].sum() * 100,
    2
))

print("\nTop 100 future visibility rate:")
print(round(
    top_100_test["future_visible"].mean() * 100,
    2
))


In [ ]:

# Step 19: Compare two historical ranking methods

# Rank by historical impressions only
impressions_top100 = test_ranked.sort_values(
    "impressions_prev30",
    ascending=False
).head(100)

# Rank by historical CTR opportunity score
opportunity_top100 = test_ranked.sort_values(
    "historical_opportunity_score",
    ascending=False
).head(100)

total_gap = test_ranked["future_click_gap"].sum()

impressions_gap = impressions_top100[
    "future_click_gap"
].sum()

opportunity_gap = opportunity_top100[
    "future_click_gap"
].sum()

print("Impressions-only top 100 capture (%):",
      round(impressions_gap / total_gap * 100, 2))

print("Opportunity-score top 100 capture (%):",
      round(opportunity_gap / total_gap * 100, 2))

print("\nDifference (percentage points):")
print(round(
    (opportunity_gap - impressions_gap) /
    total_gap * 100,
    2
))

print("\nShared records between the two top-100 lists:")
print(len(
    set(impressions_top100.index) &
    set(opportunity_top100.index)
))


In [ ]:

# Step 20: Examine future click-gap target

train_data = candidates.iloc[train_idx].copy()

train_data["future_click_gap"] = (
    0.02 * train_data["impressions_last30"]
    - train_data["clicks_last30"]
).clip(lower=0)

print("Training records:", len(train_data))

print("\nFuture click-gap summary:")
print(
    train_data["future_click_gap"]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99])
    .round(2)
)

print("\nRecords with zero future click gap:")
print((train_data["future_click_gap"] == 0).sum())

print("\nRecords with positive future click gap:")
print((train_data["future_click_gap"] > 0).sum())


In [ ]:

# Step 21: Train a Random Forest click-gap model

from sklearn.ensemble import RandomForestRegressor
import numpy as np

# Training target: future click gap
y_gap_train = train_data["future_click_gap"]

# Log-transform the skewed target
y_gap_train_log = np.log1p(y_gap_train)

# Use a smaller sample for the first experiment
rng = np.random.RandomState(42)
sample_positions = rng.choice(
    len(X_train),
    size=min(100000, len(X_train)),
    replace=False
)

X_train_sample = X_train.iloc[sample_positions]
y_train_sample = y_gap_train_log.iloc[sample_positions]

gap_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

gap_model.fit(X_train_sample, y_train_sample)

print("Random Forest training complete!")
print("Training sample:", len(X_train_sample))
print("Features used:", X_train_sample.shape[1])
print("Trees trained:", gap_model.n_estimators)


In [ ]:

# Step 22: Evaluate Random Forest ranking on held-out clients

# Predict log-transformed future click gaps
predicted_log_gap = gap_model.predict(X_test)

# Convert predictions back to the original scale
predicted_gap = np.expm1(predicted_log_gap)

# Attach predictions to held-out test records
ml_ranked = test_ranked.copy()

ml_ranked["predicted_click_gap"] = predicted_gap

# Select the 100 highest-ranked ML opportunities
ml_top100 = (
    ml_ranked
    .sort_values("predicted_click_gap", ascending=False)
    .head(100)
)

# Evaluate using actual future click gaps
ml_captured_gap = ml_top100["future_click_gap"].sum()

ml_capture_pct = (
    ml_captured_gap /
    ml_ranked["future_click_gap"].sum() * 100
)

print("ML top-100 future click gap captured:",
      round(ml_captured_gap, 2))

print("ML top-100 capture percentage:",
      round(ml_capture_pct, 2))

print("\nComparison:")
print("Impressions-only baseline: 22.88%")
print("Opportunity-score baseline: 22.72%")
print("Random Forest:", round(ml_capture_pct, 2), "%")


In [ ]:

# Step 23: Correct prediction-to-record alignment

# Start with test records in their original order
ml_ranked_corrected = candidates.iloc[test_idx].copy()

# Calculate actual future click gap
ml_ranked_corrected["future_click_gap"] = (
    0.02 * ml_ranked_corrected["impressions_last30"]
    - ml_ranked_corrected["clicks_last30"]
).clip(lower=0)

# Predictions correspond to X_test in this same row order
ml_ranked_corrected["predicted_click_gap"] = (
    np.expm1(gap_model.predict(X_test))
)

# Sort only AFTER predictions have been assigned
ml_top100_corrected = (
    ml_ranked_corrected
    .sort_values("predicted_click_gap", ascending=False)
    .head(100)
)

total_gap = ml_ranked_corrected["future_click_gap"].sum()
captured_gap = ml_top100_corrected["future_click_gap"].sum()

print("Corrected ML top-100 gap:", round(captured_gap, 2))
print("Corrected ML capture (%):",
      round(captured_gap / total_gap * 100, 2))

print("\nBaseline comparison:")
print("Impressions only: 22.88%")
print("Opportunity score: 22.72%")


In [ ]:

# Step 24: Examine Random Forest feature importance

import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gap_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

print("Random Forest feature importance:")
print(feature_importance.to_string(index=False))

print("\nTotal importance:")
print(round(feature_importance["Importance"].sum(), 4))


In [ ]:

# Step 25: Compare ML recommendations with baselines

ml_ids = set(ml_top100_corrected.index)
impressions_ids = set(impressions_top100.index)
opportunity_ids = set(opportunity_top100.index)

print("Shared recommendations:")

print(
    "ML vs impressions-only:",
    len(ml_ids & impressions_ids),
    "out of 100"
)

print(
    "ML vs opportunity score:",
    len(ml_ids & opportunity_ids),
    "out of 100"
)

print(
    "\nUnique ML recommendations vs impressions-only:",
    len(ml_ids - impressions_ids)
)

print(
    "Unique ML recommendations vs opportunity score:",
    len(ml_ids - opportunity_ids)
)


In [ ]:

# Step 26: Compare rankings at different review budgets

ranking_methods = {
    "Random Forest": (
        ml_ranked_corrected.sort_values(
            "predicted_click_gap", ascending=False
        )
    ),
    "Impressions only": (
        test_ranked.sort_values(
            "impressions_prev30", ascending=False
        )
    ),
    "Opportunity score": (
        test_ranked.sort_values(
            "historical_opportunity_score", ascending=False
        )
    )
}

total_gap = test_ranked["future_click_gap"].sum()

results = []

for k in [20, 50, 100, 200, 500]:
    row = {"Top K": k}

    for name, ranked_df in ranking_methods.items():
        captured = ranked_df.head(k)[
            "future_click_gap"
        ].sum()

        row[name] = round(
            captured / total_gap * 100, 2
        )

    results.append(row)

ranking_comparison = pd.DataFrame(results)

print("Future click-gap capture (%):")
print(ranking_comparison.to_string(index=False))


In [ ]:

# Step 27: Create a research-ready results table

results_table = ranking_comparison.copy()

results_table = results_table.rename(columns={
    "Top K": "top_k",
    "Random Forest": "random_forest_capture_pct",
    "Impressions only": "impressions_capture_pct",
    "Opportunity score": "opportunity_capture_pct"
})

results_table["ml_vs_impressions_pp"] = (
    results_table["random_forest_capture_pct"]
    - results_table["impressions_capture_pct"]
).round(2)

print("Final exploratory ranking results:")
print(results_table.to_string(index=False))

# Save results as a CSV file
results_table.to_csv(
    "ranking_comparison.csv",
    index=False
)

print("\nSaved: ranking_comparison.csv")


In [ ]:

# Step 28: Examine client contributions to test results

client_summary = (
    ml_ranked_corrected
    .groupby("client_hash_id")
    .agg(
        records=("future_click_gap", "size"),
        future_click_gap=("future_click_gap", "sum")
    )
    .reset_index()
)

client_summary["gap_share_pct"] = (
    client_summary["future_click_gap"] /
    client_summary["future_click_gap"].sum() * 100
)

client_summary = client_summary.sort_values(
    "future_click_gap",
    ascending=False
)

# Use anonymous client numbers for display
client_summary["client"] = [
    f"Client {i+1}"
    for i in range(len(client_summary))
]

print("Held-out client contributions:")
print(
    client_summary[
        ["client", "records", "future_click_gap", "gap_share_pct"]
    ].round(2).to_string(index=False)
)

print("\nLargest client's share of future click gap (%):")
print(round(client_summary["gap_share_pct"].max(), 2))


In [ ]:

# Step 29: Compare ranking methods within each client

client_results = []

for client_id, group in ml_ranked_corrected.groupby(
    "client_hash_id"
):
    # Skip clients with fewer than 20 records
    if len(group) < 20:
        continue

    total_gap = group["future_click_gap"].sum()

    if total_gap == 0:
        continue

    ml_top20 = group.nlargest(
        20, "predicted_click_gap"
    )

    impressions_top20 = group.nlargest(
        20, "impressions_prev30"
    )

    ml_capture = (
        ml_top20["future_click_gap"].sum()
        / total_gap * 100
    )

    impressions_capture = (
        impressions_top20["future_click_gap"].sum()
        / total_gap * 100
    )

    client_results.append({
        "client_id": client_id,
        "records": len(group),
        "ml_capture_pct": round(ml_capture, 2),
        "impressions_capture_pct": round(
            impressions_capture, 2
        ),
        "ml_advantage_pp": round(
            ml_capture - impressions_capture, 2
        )
    })

client_comparison = pd.DataFrame(client_results)

# Display anonymous labels
client_comparison["client"] = [
    f"Client {i+1}"
    for i in range(len(client_comparison))
]

print("Client-level Top 20 comparison:")
print(
    client_comparison[
        [
            "client",
            "records",
            "ml_capture_pct",
            "impressions_capture_pct",
            "ml_advantage_pp"
        ]
    ].to_string(index=False)
)

print("\nClients where ML performs better:")
print((client_comparison["ml_advantage_pp"] > 0).sum())

print("\nClients where impressions-only performs better:")
print((client_comparison["ml_advantage_pp"] < 0).sum())


In [ ]:

# Step 30: Summarize client-level ML performance

advantages = client_comparison["ml_advantage_pp"]

print("Clients evaluated:", len(advantages))

print("\nAverage ML advantage (percentage points):")
print(round(advantages.mean(), 2))

print("\nMedian ML advantage (percentage points):")
print(round(advantages.median(), 2))

print("\nBest client ML advantage:")
print(round(advantages.max(), 2))

print("\nWorst client ML advantage:")
print(round(advantages.min(), 2))

print("\nML wins:")
print((advantages > 0).sum())

print("\nImpressions-only wins:")
print((advantages < 0).sum())


In [ ]:

# Step 31: Save client-level evaluation results

client_results_export = client_comparison[
    [
        "client",
        "records",
        "ml_capture_pct",
        "impressions_capture_pct",
        "ml_advantage_pp"
    ]
].copy()

client_results_export.to_csv(
    "client_level_comparison.csv",
    index=False
)

print("Saved: client_level_comparison.csv")

print("\nClient-level summary:")
print("Clients evaluated:", len(client_results_export))
print(
    "Average ML advantage:",
    round(client_results_export["ml_advantage_pp"].mean(), 2),
    "percentage points"
)
print(
    "Median ML advantage:",
    round(client_results_export["ml_advantage_pp"].median(), 2),
    "percentage points"
)

print("\nCSV preview:")
print(client_results_export.head().to_string(index=False))



## 3. Results and Discussion

### 3.1 Global Ranking Performance

Three ranking methods were evaluated:

1. Historical impressions-only ranking
2. Historical CTR opportunity-score ranking
3. Random Forest regression

The evaluation used 41,742 records from 10 held-out clients.

At Top 100, Random Forest captured 22.90% of the
observed future click gap, compared with 22.88%
for impressions-only and 22.72% for the
historical opportunity score.

However, Random Forest did not consistently
outperform the simpler baselines across review
budgets of 20, 50, 100, 200, and 500 records.

### 4.2 Feature Importance

Historical impressions contributed 84.49%
of the Random Forest's impurity-based
feature importance.

This suggests that impression volume was
the dominant feature in the fitted model.

Feature importance does not establish causation.

### 4.3 Client-Level Evaluation

Eight clients with at least 20 eligible records
were evaluated separately.

Random Forest outperformed impressions-only
ranking for five clients and underperformed
for three clients.

The average ML advantage was -0.83 percentage
points, while the median was +0.38 percentage
points.

The largest disadvantage was -10.55 percentage
points for one client.

### 4.4 Preliminary Interpretation

The Random Forest produced different
recommendations from the rule-based baselines,
but did not demonstrate a consistent improvement
in future click-gap capture.

The results suggest that simpler ranking
methods remain competitive for this task.

These findings are exploratory and do not
establish that identified click gaps can
be recovered through SEO interventions.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations


### 5.1 Client Imbalance

The evaluation included 10 held-out clients.
However, the three largest contributors accounted
for 96.36% of the total future click gap.

This means the overall results were strongly
influenced by a small number of clients.

### 5.2 Limited Generalization

The experiment used one client-group train/test
split, with no overlapping clients.

Results were not validated across multiple
independent splits. Therefore, performance
on other clients remains uncertain.

### 5.3 Historical Data Limitations

The dataset covered one 90-day reporting window.

Historical search signals from the previous
30-day period were used to predict outcomes
in the latest 30-day period.

Performance across other time periods and
seasons was not evaluated.

### 5.4 CTR Threshold Assumption

A 2% CTR reference was used to calculate
future click gaps.

This threshold was a research assumption,
not evidence that every search query could
achieve a 2% CTR.

The calculated gaps do not represent
guaranteed additional clicks.

### 5.5 Machine Learning Limitations

The Random Forest model was trained using
100,000 sampled training records and six
historical features.

The model did not consistently outperform
the simpler ranking methods.

Repeated exploratory evaluation on the same
held-out test set also limits the strength
of conclusions about model superiority.

### 5.6 No Causal SEO Evaluation

The study did not implement or experimentally
test SEO improvements.

It evaluated the ability of historical
search signals to rank records associated
with future CTR shortfalls.

Therefore, the findings cannot establish
that acting on these recommendations would
increase website traffic.



### 5.7 Search Position Data Quality

The historical search-position feature contained
unexpected values.

Among 390,353 eligible candidate records,
7,942 records (2.03%) had an average historical
position below 1, including 907 records with
a position equal to zero.

Under the standard Google Search Console
position definition, values below 1 are
unexpected.

The cause could not be established from the
available anonymized dataset.

One of the top 20 ranked recommendations
contained an average historical position
of approximately 0.32.

Rather than deleting or modifying the record,
the recommendation was flagged for manual
data-quality review.

The original ranking was preserved.

This issue limits confidence in interpretations
that rely on the historical position feature.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:

# Step 34: Generate ranked SEO review candidates

recommendations = (
    ml_ranked_corrected
    .sort_values("impressions_prev30", ascending=False)
    .head(20)
    .copy()
)

recommendations["historical_ctr_pct"] = (
    recommendations["ctr_prev30"] * 100
)

recommendations["recommended_action"] = (
    "Review search snippet, title, and query intent"
)

recommendations["rank"] = range(
    1, len(recommendations) + 1
)

columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "query_hash_id",
    "impressions_prev30",
    "historical_ctr_pct",
    "recommended_action"
]

print("Top 20 SEO review candidates:")
print(
    recommendations[columns]
    .round(2)
    .to_string(index=False)
)

recommendations[columns].to_csv(
    "ranked_recommendations.csv",
    index=False
)

print("\nSaved: ranked_recommendations.csv")


In [ ]:

# Step 35: Add evidence-based review reasons

import numpy as np

recommendations["review_reason"] = np.select(
    [
        (recommendations["ctr_prev30"] < 0.005) &
        (recommendations["avg_position_prev30"] <= 10),

        recommendations["ctr_prev30"] < 0.005
    ],
    [
        "Very low CTR despite average top-10 position",
        "Very low CTR; investigate ranking and query intent"
    ],
    default="High impressions; review CTR and search appearance"
)

display_columns = [
    "rank",
    "impressions_prev30",
    "historical_ctr_pct",
    "avg_position_prev30",
    "review_reason"
]

print("Top 20 recommendations with review reasons:")
print(
    recommendations[display_columns]
    .round(2)
    .to_string(index=False)
)

# Update the exported recommendations
export_columns = columns + [
    "avg_position_prev30",
    "review_reason"
]

recommendations[export_columns].to_csv(
    "ranked_recommendations.csv",
    index=False
)

print("\nUpdated: ranked_recommendations.csv")


In [ ]:

# Step 36: Validate historical search-position values

position = candidates["avg_position_prev30"]

print("Historical position summary:")
print(position.describe().round(2))

print("\nRecords with position below 1:")
print((position < 1).sum())

print("\nRecords with position equal to 0:")
print((position == 0).sum())

print("\nRecords with missing position:")
print(position.isna().sum())

print("\nTop 20 recommendations with position below 1:")
print(
    (recommendations["avg_position_prev30"] < 1).sum()
)


In [ ]:

# Step 37: Flag unexpected position values

recommendations["position_quality"] = np.where(
    recommendations["avg_position_prev30"] < 1,
    "Review: position below 1",
    "Within expected range"
)

recommendations["review_reason"] = np.where(
    recommendations["avg_position_prev30"] < 1,
    "Validate position data before SEO diagnosis",
    recommendations["review_reason"]
)

print("Position quality in top 20:")
print(recommendations["position_quality"].value_counts())

print("\nFlagged recommendations:")
print(
    recommendations.loc[
        recommendations["avg_position_prev30"] < 1,
        [
            "rank",
            "impressions_prev30",
            "avg_position_prev30",
            "review_reason"
        ]
    ].to_string(index=False)
)

# Save the updated recommendations
export_columns = columns + [
    "avg_position_prev30",
    "review_reason",
    "position_quality"
]

recommendations[export_columns].to_csv(
    "ranked_recommendations.csv",
    index=False
)

print("\nUpdated: ranked_recommendations.csv")


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:

# Step 39: Create ranking comparison chart

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    ranking_comparison["Top K"],
    ranking_comparison["Random Forest"],
    marker="o",
    label="Random Forest"
)

plt.plot(
    ranking_comparison["Top K"],
    ranking_comparison["Impressions only"],
    marker="s",
    label="Impressions only"
)

plt.plot(
    ranking_comparison["Top K"],
    ranking_comparison["Opportunity score"],
    marker="^",
    label="Opportunity score"
)

plt.xlabel("Number of recommendations (Top K)")
plt.ylabel("Future click-gap capture (%)")
plt.title("Ranking Performance on Held-Out Clients")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "ranking_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: ranking_comparison.png")


In [ ]:

# Step 40: Create feature importance chart

import pandas as pd
import matplotlib.pyplot as plt

importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gap_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=True
)

plt.figure(figsize=(9, 5))

plt.barh(
    importance_df["Feature"],
    importance_df["Importance"] * 100
)

plt.xlabel("Feature importance (%)")
plt.title("Random Forest Feature Importance")
plt.tight_layout()

plt.savefig(
    "feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: feature_importance.png")


In [ ]:

# Step 41: Visualize client-level ML performance

import matplotlib.pyplot as plt

chart_data = client_results_export.copy()

plt.figure(figsize=(9, 5))

bars = plt.bar(
    chart_data["client"],
    chart_data["ml_advantage_pp"],
    color=[
        "green" if value > 0 else "red"
        for value in chart_data["ml_advantage_pp"]
    ]
)

plt.axhline(0, color="black", linewidth=1)

plt.xlabel("Held-out client")
plt.ylabel("ML advantage (percentage points)")
plt.title("Random Forest vs Impressions-Only by Client")

plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    "client_level_performance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: client_level_performance.png")


In [ ]:

# Step 43: Package research artifacts

import zipfile
import os

artifact_files = [
    "ranking_comparison.csv",
    "client_level_comparison.csv",
    "ranked_recommendations.csv",
    "ranking_comparison.png",
    "feature_importance.png",
    "client_level_performance.png"
]

with zipfile.ZipFile(
    "capstone_artifacts.zip", "w"
) as zip_file:
    for filename in artifact_files:
        if os.path.exists(filename):
            zip_file.write(filename)
            print("Added:", filename)
        else:
            print("MISSING:", filename)

print("\nCreated: capstone_artifacts.zip")



## 8. Research Paper Summary

### Title
Predicting Google Search Engagement Opportunities
Using Machine Learning

### Abstract

This study investigates whether historical Google
Search performance signals can identify records
associated with future click-through rate (CTR)
shortfalls and whether machine learning improves
opportunity ranking compared with simpler methods.

The experiment used anonymized Google Search
performance data and evaluated 41,742 records
from 10 held-out clients.

Three ranking approaches were compared:
historical impressions-only ranking,
a CTR opportunity-score baseline,
and Random Forest regression.

At Top 100, Random Forest captured 22.90% of the
observed future click gap, compared with 22.88%
for impressions-only and 22.72% for the
opportunity-score baseline.

However, Random Forest did not consistently
outperform simpler approaches across different
review budgets or clients.

The findings suggest that historical search
impressions provide a strong baseline for
prioritizing CTR-related review candidates.

The study measures observational CTR shortfalls,
not actual traffic gains from SEO interventions.
Further validation is needed before deployment.



### 1. Introduction

Google Search performance data provides
information about how frequently content
appears in search results and how often
users click on it.

However, high search visibility does not
always translate into high engagement.

Content receiving many impressions but
relatively few clicks may warrant further
investigation by SEO teams.

Identifying which records deserve attention
is challenging when datasets contain
millions of observations.

Traditional approaches often prioritize
content using simple metrics such as
impressions and click-through rate (CTR).

Machine learning offers an alternative
approach by combining multiple historical
signals to estimate future outcomes.

This study investigates whether machine
learning can improve the ranking of
CTR-related review candidates compared
with simpler historical ranking methods.

### 1.1 Research Question

Can historical Google Search performance
signals help identify records associated
with future low-CTR outcomes, and can
machine learning improve opportunity
ranking compared with rule-based baselines?

### 1.2 Research Objectives

1. Analyze historical Google Search
   impressions, clicks, CTR, and position.

2. Develop simple historical ranking
   baselines for identifying review candidates.

3. Train a Random Forest regression model
   using historical search performance features.

4. Compare machine learning and baseline
   rankings using future click-gap capture
   at different Top-K review budgets.

5. Examine performance differences across
   held-out clients.

6. Generate ranked SEO review recommendations
   while documenting data-quality concerns
   and methodological limitations.

### 1.3 Research Contribution

This study provides an empirical comparison
between a machine learning ranking model
and simpler historical ranking approaches.

It emphasizes reproducible evaluation,
client-level analysis, and transparent
reporting of limitations.

The study does not claim that predicted
CTR shortfalls represent guaranteed
additional traffic.



### 2. Dataset Description

#### 2.1 Data Source

This study used anonymized Google Search
performance data from the FlyRank
internship warehouse.

The dataset was obtained from the
`fact_content_query_90d.parquet` file.

It contained 2,414,248 records and
21 columns representing search performance
across 52 anonymized clients.

The reporting window covered
April 2 to June 30, 2026.

Each record represented a client,
content item, and search query combination.

#### 2.2 Dataset Features

The dataset included:

- Historical search impressions and clicks
- Previous and latest 30-day performance
- Average search position
- Query character and token counts
- Content-level visibility statistics
- Anonymized client, content, and query IDs

Only historical information was used
as input to the predictive model.

#### 2.3 Candidate Selection

Records were selected for modeling when
they had at least 30 impressions during
the previous 30-day period.

This produced 390,353 candidate records.

Unlike filtering on future visibility,
this selection rule relied only on
historical information available at
the time of prediction.

#### 2.4 Feature Engineering

Six historical features were used:

1. `impressions_prev30`
2. `clicks_prev30`
3. `avg_position_prev30`
4. `ctr_prev30`
5. `query_token_count`
6. `query_char_count`

Historical CTR was calculated as:

CTR = Previous 30-day clicks /
Previous 30-day impressions

#### 2.5 Prediction Target

The target was a future click gap,
defined as:

Future click gap =
max(0, 0.02 × latest 30-day impressions
- latest 30-day clicks)

The 2% CTR reference was a research
assumption used to compare ranking methods.

It was not an estimate of guaranteed
recoverable clicks.

#### 2.6 Training and Test Split

A group-based train/test split was
performed using anonymized client IDs.

The training set contained 348,611 records
from 38 clients.

The test set contained 41,742 records
from 10 held-out clients.

There was no client overlap between
training and testing.

This design evaluated generalization
to unseen clients rather than
future reporting periods.



### 3. Research Methodology

#### 3.1 Research Design

This study used a retrospective,
quantitative machine learning experiment
to compare three methods for ranking
Google Search engagement review candidates.

The methods were evaluated using
historical search features and observed
outcomes from a later 30-day period.

#### 3.2 Baseline Ranking Methods

Two simple ranking methods were evaluated.

**Baseline 1: Impressions-only**

Records were ranked by previous 30-day
impressions in descending order.

This method prioritizes records with
the greatest historical search visibility.

**Baseline 2: Historical CTR Opportunity Score**

An opportunity score was calculated as:

Opportunity score =
Previous impressions ×
max(0, 0.02 - Historical CTR)

This method combines historical search
visibility with a CTR shortfall relative
to the chosen 2% reference.

#### 3.3 Machine Learning Model

A Random Forest regression model was
trained using six historical features.

The model used 100 decision trees and
was trained on a random sample of
100,000 training records.

The prediction target was the observed
future click gap relative to the
2% CTR reference.

Predicted values were used to rank
candidate records for review.

#### 3.4 Evaluation Strategy

The study used a client-group split
to prevent the same client from
appearing in both training and testing.

All three ranking methods were
evaluated on the same held-out
test records.

The primary evaluation metric was
future click-gap capture at Top K.

Capture at K was calculated as:

Capture@K (%) =
100 ×
(Sum of observed future click gaps
among the top K ranked records) /
(Sum of observed future click gaps
across all test records)

Evaluation budgets included:

- Top 20
- Top 50
- Top 100
- Top 200
- Top 500

Higher capture indicates that a
ranking method placed more observed
future click-gap units within its
selected review budget.

#### 3.5 Client-Level Evaluation

Ranking performance was also
examined separately for clients
with at least 20 candidate records.

For each eligible client, the
top 20 records were selected using
Random Forest predictions and
historical impressions.

The percentage-point difference
between the methods was calculated.

#### 3.6 Model Interpretation

Random Forest impurity-based feature
importance was examined to understand
which historical features contributed
most to the fitted model's splits.

These importance values were treated
as descriptive rather than causal.

#### 3.7 Exploratory Evaluation

The held-out test set was examined
repeatedly during the investigation.

Therefore, comparisons were treated
as exploratory rather than as an
independent final assessment of
model superiority.



### 4. Results and Discussion

#### 4.1 Global Ranking Performance

The three ranking methods were evaluated
on 41,742 records from 10 held-out clients.

The total observed future click gap across
the test records was 122,355.20 units.

The table below reports the percentage of
future click gap captured at different
recommendation budgets.

| Top K | Random Forest | Impressions-only | Opportunity Score |
|---|---:|---:|---:|
| 20 | 10.60% | 15.66% | 15.66% |
| 50 | 19.28% | 19.25% | 19.29% |
| 100 | 22.90% | 22.88% | 22.72% |
| 200 | 25.76% | 26.85% | 26.85% |
| 500 | 31.91% | 33.52% | 33.04% |

At Top 100, Random Forest achieved the
highest capture rate at 22.90%.

However, this represented only a
0.02 percentage-point improvement over
the impressions-only baseline.

At Top 20, Random Forest performed
5.06 percentage points worse than
both simple baselines.

The impressions-only baseline also
outperformed Random Forest at
Top 200 and Top 500.

These findings indicate that Random Forest
did not consistently improve ranking
performance.

#### 4.2 Random Forest Feature Importance

Historical impressions were the dominant
feature in the fitted Random Forest model.

The impurity-based feature importance
values were:

| Historical Feature | Importance |
|---|---:|
| Impressions | 84.49% |
| Average search position | 7.74% |
| Query character count | 3.62% |
| Historical CTR | 2.33% |
| Query token count | 1.45% |
| Historical clicks | 0.37% |

The strong contribution of historical
impressions helps explain why the
Random Forest and impressions-only
baseline produced similar results.

These importance values do not
establish causal relationships.

#### 4.3 Client-Level Ranking Performance

Eight held-out clients had at least
20 eligible candidate records.

Random Forest performed better than
impressions-only ranking for five clients
and worse for three clients.

The average ML advantage was
-0.83 percentage points.

The median ML advantage was
+0.38 percentage points.

The largest ML disadvantage was
-10.55 percentage points for
one client with 14,306 records.

Although Random Forest won for more
clients than it lost, its improvements
were generally small and its performance
was inconsistent.

#### 4.4 Client Concentration

The three largest contributors accounted
for 96.36% of the total future click gap
in the held-out test set.

This means the global ranking results
were strongly influenced by a small
number of clients.

Client-level results therefore provide
important additional context.

#### 4.5 Interpretation

The experiment demonstrates that
historical Google Search signals can
be used to rank records associated
with observed future CTR shortfalls.

However, the Random Forest model
did not demonstrate a reliable
advantage over simple ranking methods.

For this dataset and evaluation setup,
historical impressions-only ranking
remains a competitive and interpretable
baseline.

The results are exploratory and
should not be interpreted as evidence
of additional traffic achievable
through SEO changes.



### 5. Research Limitations

#### 5.1 Client Imbalance

The held-out test set contained 10 clients,
but three clients accounted for 96.36%
of the total observed future click gap.

Consequently, the global evaluation was
strongly influenced by a small number
of high-volume clients.

#### 5.2 Generalization Limitations

The experiment used one client-group
train/test split.

Although training and test clients
did not overlap, the study did not
evaluate multiple independent splits.

Therefore, the stability of the findings
across other client groups remains uncertain.

#### 5.3 Temporal Limitations

The dataset covered one 90-day
reporting window.

Previous 30-day search signals were
used to predict outcomes in the
latest 30-day period.

Performance across different seasons
or reporting windows was not assessed.

#### 5.4 CTR Reference Assumption

The study used a 2% CTR reference
to calculate future click gaps.

This threshold was selected for
research purposes and was not
established as an achievable CTR
for every search query.

The calculated click gaps should not
be interpreted as recoverable traffic.

#### 5.5 Model Limitations

The Random Forest model used six
historical features and 100 trees.

It was trained on a sample of
100,000 training records.

The experiment did not establish
that the selected model configuration
was optimal.

Repeated exploratory analysis on
the same held-out test set also
limits the strength of conclusions
about model superiority.

#### 5.6 Search Position Data Quality

Among 390,353 candidate records,
7,942 records (2.03%) had historical
average search positions below 1.

These values are unexpected under
the standard Google Search Console
position definition.

One of the top 20 recommendations
had a reported position of 0.32.

This record was flagged for
manual data-quality review rather
than being deleted or modified.

#### 5.7 Lack of Causal Validation

The study did not implement or
experimentally evaluate SEO changes.

Therefore, the results demonstrate
ranking performance against observed
future CTR shortfalls, not actual
improvements in website traffic.



### 6. Conclusion and Future Work

#### 6.1 Conclusion

This study investigated whether historical
Google Search performance signals could
help identify content engagement review
opportunities and whether machine learning
could improve opportunity ranking compared
with simple historical baselines.

Three approaches were evaluated:
impressions-only ranking, a historical
CTR opportunity score, and Random Forest
regression.

The experiment used 41,742 test records
from 10 held-out clients.

At Top 100, Random Forest captured 22.90%
of the observed future click gap, compared
with 22.88% for impressions-only ranking
and 22.72% for the opportunity-score baseline.

However, Random Forest did not consistently
outperform the simpler approaches across
different recommendation budgets.

Client-level evaluation also revealed
variation in model performance, including
a substantial disadvantage for one client.

Historical impressions accounted for
84.49% of the Random Forest's impurity-based
feature importance.

Overall, the findings suggest that
historical search signals can support
the prioritization of CTR-related review
candidates, but the tested machine learning
approach did not demonstrate a consistent
advantage over simpler ranking methods.

The study evaluated observational
CTR shortfalls and did not establish
that the identified gaps could be
recovered through SEO interventions.

#### 6.2 Future Work

Future research could improve this
approach in several ways:

1. **Repeated client-group validation:**
   Evaluate ranking performance across
   multiple independent client splits.

2. **Temporal validation:**
   Test models on additional reporting
   periods to assess stability over time.

3. **Alternative models:**
   Compare Random Forest with gradient
   boosting and other learning-to-rank
   approaches.

4. **Client-level ranking:**
   Develop ranking strategies that
   allocate review opportunities
   separately for each client.

5. **Data-quality investigation:**
   Verify the definition and processing
   of search-position values below 1.

6. **CTR benchmarking:**
   Explore reference CTR values adjusted
   for search position and query context.

7. **Real-world SEO evaluation:**
   Test selected recommendations through
   controlled SEO experiments to measure
   actual changes in clicks and engagement.

#### 6.3 Final Research Contribution

This project demonstrates a reproducible
workflow for ranking Google Search
engagement review candidates using
historical signals.

Its main contribution is an evidence-based
comparison showing that increased model
complexity does not necessarily produce
better ranking performance.

The study also highlights the importance
of strong baselines, client-level analysis,
data-quality checks, and transparent
reporting of limitations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
